# 情景记忆（Episodic Memory）

> **把完整的交互片段连同时间上下文一起存下来，让智能体跨会话回忆起「什么时候发生了什么」。**

你不会把自己的生活记成一张平铺的事实清单。你记住的是*一个个片段（episode）*：一段有时间和地点的完整经历。「决定产品转型的那次会议。」「耗掉一整个下午的调试。」每个片段都有开头、经过和结尾。

情景记忆把这种能力带给 AI 智能体。它把整段对话会话（或有意义的部分）保存成一个个独立的片段，每个片段都带上时间戳、主题标签和自动生成的摘要。这样，智能体回忆过往经历时，得到的是连贯的事件，而不是孤立的事实。

这一点很重要，因为很多智能体的使用场景都横跨多个会话。教练型智能体需要想起上周定了什么目标；项目管理智能体需要知道三天前做了什么决定。没有情景记忆，每个会话都从零开始，智能体在两次交互之间是失忆的。

核心工程挑战是**片段边界检测**（判断一个片段在哪里结束、下一个片段从哪里开始）。边界可以按会话划分（一个会话 = 一个片段）、按主题划分（检测到主题切换就开启新片段），或按时间划分（间隔 *n* 分钟就开启新片段）。第二个挑战是检索：拿到一个新的用户查询，怎么找到最相关的过往片段？这需要把时序索引、语义相似度和显式指代检测结合起来。

**读完本 notebook 你会理解：**
- 如何把对话轮次连同元数据存成结构化片段。
- 如何检测片段边界（按会话和按主题两种方式）。
- 如何用 LLM 生成片段摘要。
- 如何结合新近度和语义相似度检索相关片段。
- 情景记忆什么时候有帮助，什么时候会悄悄失效。


## 关键概念

- **片段（episode）**：一段有边界的对话，作为一个整体存储。可以把它想成日记里的一章。它有开始时间、结束时间、一份消息列表和一个摘要。
- **片段边界**：两个片段之间的分界线。边界可以按会话划分（一个会话 = 一个片段）、按主题划分（检测到主题切换就开启新片段），或按时间划分（间隔 *n* 分钟就开启新片段）。
- **时序索引**：给每个片段打上时间戳（开始时间、结束时间、时长）。这让你能做「上周二我们讨论了什么？」「上一个会话发生了什么？」这类基于时间的查询。
- **片段摘要**：概括片段大意的一段短文本。片段关闭时，由 LLM（large language model，生成文本的 AI）生成这段摘要。有了摘要，你不用加载完整对话记录就能快速浏览过往片段。
- **嵌入（embedding）**：一串表示文本含义的数字。含义相近的两段文本，其嵌入在向量空间中的位置也彼此接近。我们用嵌入来找到与查询语义相关的片段。
- **余弦相似度**：衡量两个嵌入有多接近的方法。取值范围从 -1（方向相反）到 1（完全相同），值越大说明两段文本越相关。
- **情景记忆 vs. 语义记忆**：情景记忆存储具体经历（「3 月 5 日用户让我重写引言」）。语义记忆存储提炼后的知识（「用户喜欢简洁的文风」）。两者都有价值，但角色不同。
- **片段衰减**：在长期运行的智能体里，较老的片段会被压缩或修剪，让存储保持在可管理的规模。


## Architecture

<p align="center">
  <img src="../../images/diagrams/09_episodic_memory.svg" alt="情景记忆架构图" width="720"/>
</p>

<details><summary>Mermaid source</summary>

```mermaid
flowchart LR
    Stream["Conversation\nStream"] --> Detector["Episode Boundary\nDetector"]
    Detector -- "episode complete" --> Packager["Episode Packager\n(metadata + summary)"]
    Packager --> Store["Episode Store\n(indexed by time, topic)"]
    Detector -- "continuing" --> Stream
    NewQuery["New User Query"] --> Retrieval["Retrieval Engine\n(temporal + semantic)"]
    Store --> Retrieval
    Retrieval --> Injection["Context Injection\n(episode summaries)"]
    Injection --> LLM["LLM"]
    LLM --> Response["Response"]
```

</details>

**数据流：** 对话流经过片段边界检测器。当一个片段完成（会话结束、主题切换或时间间隔触发），片段打包器会生成一条结构化记录，包含时间戳、主题标签和 LLM 生成的摘要。这条记录进入片段库，同时支持时序检索和语义检索。新查询到来时，检索引擎找出相关的过往片段，把它们的摘要注入 LLM 提示词。

## 环境准备

安装依赖并配置 API 访问。你需要在一个 `.env` 文件里设置 `LLM_API_KEY` 环境变量。


In [ ]:
%pip install -q python-dotenv numpy deepagents langchain-openai

导入库并初始化 OpenAI 客户端。API 密钥从 `.env` 文件加载。


In [ ]:
import os
import json
import uuid
import copy
from datetime import datetime, timedelta
from dataclasses import dataclass, field, asdict
import numpy as np

from dotenv import load_dotenv

load_dotenv()  # 从 .env 读取 LLM_API_KEY / LLM_BASE_URL / LLM_MODEL_NAME

from langchain_openai import ChatOpenAI
from deepagents import create_deep_agent

assert os.getenv("LLM_API_KEY"), "Set LLM_API_KEY in your .env file"

# 任何 OpenAI 兼容端点都可以：用官方 OpenAI 时省略 base_url 即可
llm = ChatOpenAI(
    model=os.getenv("LLM_MODEL_NAME", "gpt-4o-mini"),
    api_key=os.getenv("LLM_API_KEY"),
    base_url=os.getenv("LLM_BASE_URL"),
)
def to_text(content) -> str:
    """把字符串或内容块列表统一转成纯文本。"""
    if isinstance(content, str):
        return content
    return "".join(
        block.get("text", "") for block in content if isinstance(block, dict)
    )


print("✓ Model ready")


网关没有 `/embeddings` 接口时，用这份本地确定性向量兜底：词加字符 n-gram 哈希后归一化。配置 `LLM_EMBEDDING_BASE_URL` 和 `LLM_EMBEDDING_MODEL_NAME` 后，自动切换到真实向量端点。


In [ ]:
EMBEDDING_MODEL = os.getenv("LLM_EMBEDDING_MODEL_NAME", "")


def embed_texts(texts: list[str]) -> list[list[float]]:
    """把文本转成向量。

    当前网关没有 /embeddings 接口。默认退回到本地确定性向量：
    词 + 字符 n-gram 哈希后归一化，语义相近的文本仍会得到相近向量，
    足够演示记忆检索的几何逻辑。配置 LLM_EMBEDDING_BASE_URL 和
    LLM_EMBEDDING_MODEL_NAME 后，自动改用真实向量端点。
    """
    if EMBEDDING_MODEL:
        from openai import OpenAI

        client = OpenAI(
            api_key=os.getenv("LLM_EMBEDDING_API_KEY", os.getenv("LLM_API_KEY")),
            base_url=os.getenv("LLM_EMBEDDING_BASE_URL", os.getenv("LLM_BASE_URL")),
        )
        response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
        return [item.embedding for item in response.data]

    import hashlib
    import math
    import re as re_mod

    dim = 512
    vectors = []
    for text in texts:
        vec = [0.0] * dim
        for word in re_mod.findall(r"\w+", text.lower()):
            grams = {word} | {word[i:i + 3] for i in range(max(1, len(word) - 2))}
            for gram in grams:
                digest = hashlib.blake2b(gram.encode("utf-8"), digest_size=8).digest()
                index = int.from_bytes(digest, "big") % dim
                vec[index] += 1.0
        norm = math.sqrt(sum(v * v for v in vec)) or 1.0
        vectors.append([v / norm for v in vec])
    return vectors


## 核心实现

我们分四步构建情景记忆：

1. 定义 `Episode` 数据模型。
2. 构建 `EpisodicMemory` 类，包含边界检测、片段定稿和检索。
3. 把检索到的片段接入 LLM 提示词。
4. 跑一个真实的多会话示例。


### 第 1 步：Episode 数据模型

想象一个文件柜，每个文件夹装着一次会议的记录。文件夹标签上写着日期、主题和一行摘要。你可以直接抽出对的文件夹，不用翻遍里面的每一页。

我们的 `Episode` dataclass 就是这样工作的。它保存完整的对话记录，同时携带让检索变快的元数据。


In [ ]:
@dataclass
class Episode:
    """单个片段：一段有边界的对话，附带元数据。"""

    episode_id: str
    start_time: str            # ISO 格式时间戳
    end_time: str              # ISO 格式时间戳
    messages: list[dict]       # [{"role": ..., "content": ...}, ...]
    summary: str               # LLM 生成的摘要
    topic_tags: list[str]      # 抽取出的主题标签
    summary_embedding: list[float] = field(default_factory=list)  # 用于语义搜索

    def to_dict(self) -> dict:
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "Episode":
        return cls(**data)

    def __repr__(self) -> str:
        tag_str = ", ".join(self.topic_tags[:3])
        return (
            f"Episode({self.episode_id[:8]}... | "
            f"{len(self.messages)} msgs | "
            f"tags=[{tag_str}])"
        )

### 第 2 步：辅助函数

我们需要三个辅助函数，它们会调用 LLM 或嵌入 API：

1. 把一个片段的消息**总结**成一小段文字。
2. 从消息中**抽取主题标签**。
3. 为一段文本**获取嵌入**（用于语义搜索）。


In [ ]:
def summarize_messages(messages: list[dict]) -> str:
    """让 LLM 为一段对话生成简明摘要。"""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "Summarize the following conversation in 2-3 sentences. "
                    "Focus on key decisions, facts shared, and outcomes."
                ),
            },
            {"role": "user", "content": transcript},
        ])
    return to_text(response.content).strip()


def extract_topic_tags(messages: list[dict]) -> list[str]:
    """让 LLM 从一段对话中抽取 2-4 个主题标签。"""
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in messages
    )
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "Extract 2-4 short topic tags from the conversation below. "
                    "Return them as a JSON array of lowercase strings. "
                    "Example: [\"budgeting\", \"q3 goals\", \"team hiring\"]"
                ),
            },
            {"role": "user", "content": transcript},
        ])
    raw = to_text(response.content).strip()
    try:
        tags = json.loads(raw)
        if isinstance(tags, list):
            return [str(t).lower() for t in tags]
    except json.JSONDecodeError:
        pass
    # 兜底方案：JSON 解析失败时按逗号切分
    return [t.strip().lower().strip('"') for t in raw.split(",")][:4]




我们还需要嵌入和相似度函数。`get_embedding` 把文本转成向量（一串捕捉含义的数字）。`cosine_similarity` 衡量两个向量有多接近。我们用它们找出与新查询语义相关的过往片段。


In [ ]:
def get_embedding(text: str) -> list[float]:
    """获取一段文本的向量嵌入。"""
    return embed_texts([text])[0]


def cosine_similarity(vec_a: list[float], vec_b: list[float]) -> float:
    """计算两个向量的余弦相似度。"""
    a = np.array(vec_a)
    b = np.array(vec_b)
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    if norm == 0:
        return 0.0
    return float(dot / norm)

### 第 3 步：EpisodicMemory 类

这是核心类。它把消息累积进缓冲区，判断一个片段何时完成，把片段定稿（摘要 + 标签 + 嵌入）并存起来。当智能体需要上下文时，它还负责检索相关的过往片段。

我们支持两种边界策略：
- **基于会话**：你显式调用 `end_session()` 来关闭一个片段。
- **基于主题**：由 LLM 判断最新一条消息是否切换了主题。


In [ ]:
class EpisodicMemory:
    """管理情景记忆：消息累积、边界检测和检索。"""

    def __init__(self, boundary_strategy: str = "session"):
        """
        Args:
            boundary_strategy: "session"（手动）或 "topic"（LLM 检测）。
        """
        self.episodes: list[Episode] = []
        self.current_buffer: list[dict] = []
        self.current_start_time: str = datetime.now().isoformat()
        self.boundary_strategy = boundary_strategy

    # ── 添加消息 ──────────────────────────────────────────

    def add_message(self, role: str, content: str) -> bool:
        """把一条消息加入当前缓冲区。如果一个片段被定稿，返回 True。"""
        self.current_buffer.append({"role": role, "content": content})

        if self.boundary_strategy == "topic" and len(self.current_buffer) >= 4:
            if self._detect_topic_shift():
                # 最新一条消息属于新主题。
                # 把它*之前*的所有内容定稿为一个片段。
                new_topic_msg = self.current_buffer.pop()
                self._finalize_episode()
                # 用切换后的消息开启新片段
                self.current_buffer = [new_topic_msg]
                self.current_start_time = datetime.now().isoformat()
                return True
        return False



接下来添加边界检测。`_detect_topic_shift` 方法询问 LLM：最新一条消息是否换了话题。如果是，当前缓冲区就被定稿为一个片段，新片段随即开始。


In [ ]:
# ── 边界检测 ───────────────────────────────────────

def __detect_topic_shift(self) -> bool:
    """询问 LLM 最后一条消息是否切换了主题。"""
    recent = self.current_buffer[-4:]  # 取最后 4 条消息作为上下文
    transcript = "\n".join(
        f"{m['role'].upper()}: {m['content']}" for m in recent
    )
    response = llm.invoke([
            {
                "role": "system",
                "content": (
                    "You are a conversation analyst. Look at the last few messages below. "
                    "Did the LAST message introduce a significantly different topic "
                    "from what was being discussed before it? "
                    "Answer with exactly one word: YES or NO."
                ),
            },
            {"role": "user", "content": transcript},
        ])
    answer = to_text(response.content).strip().upper()
    return answer.startswith("YES")

EpisodicMemory._detect_topic_shift = __detect_topic_shift


`end_session` 方法是基于会话模式下的手动边界。`_finalize_episode` 把当前缓冲区打包成一个完整的 Episode：生成摘要、抽取主题标签、计算嵌入，然后清空缓冲区。


In [ ]:
# ── 基于会话的边界 ───────────────────────────────────

def _end_session(self) -> Episode | None:
    """手动关闭当前片段（基于会话的边界）。"""
    if not self.current_buffer:
        return None
    return self._finalize_episode()

# ── 片段定稿 ─────────────────────────────────────

def __finalize_episode(self) -> Episode:
    """把当前缓冲区打包成一个完整的 Episode。"""
    summary = summarize_messages(self.current_buffer)
    tags = extract_topic_tags(self.current_buffer)
    embedding = get_embedding(summary)

    episode = Episode(
        episode_id=str(uuid.uuid4()),
        start_time=self.current_start_time,
        end_time=datetime.now().isoformat(),
        messages=copy.deepcopy(self.current_buffer),
        summary=summary,
        topic_tags=tags,
        summary_embedding=embedding,
    )
    self.episodes.append(episode)

    # 重置缓冲区
    self.current_buffer = []
    self.current_start_time = datetime.now().isoformat()

    return episode

EpisodicMemory.end_session = _end_session

EpisodicMemory._finalize_episode = __finalize_episode


`retrieve` 方法为给定查询找出最相关的过往片段。它给每个片段打分，综合语义相似度（查询与片段摘要有多接近）和新近度（越新的片段得分越高）。你可以用 `recency_weight` 和 `semantic_weight` 控制两者的比重。


In [ ]:
# ── 检索 ────────────────────────────────────────────────

def _retrieve(
    self,
    query: str,
    k: int = 3,
    recency_weight: float = 0.3,
    semantic_weight: float = 0.7,
) -> list[tuple[Episode, float]]:
    """
    检索最相关的前 k 个片段。

    评分综合两部分：
    - 语义相似度（查询嵌入与片段摘要嵌入之间的余弦相似度）。
    - 新近度（越新的片段得分越高）。

    返回 (episode, score) 元组列表，按得分降序排列。
    """
    if not self.episodes:
        return []

    query_embedding = get_embedding(query)

    # 计算新近度得分：最新的片段得 1.0，最旧的得 0.0
    n = len(self.episodes)
    recency_scores = [i / max(n - 1, 1) for i in range(n)]

    scored = []
    for idx, episode in enumerate(self.episodes):
        sem_score = cosine_similarity(query_embedding, episode.summary_embedding)
        rec_score = recency_scores[idx]
        combined = (semantic_weight * sem_score) + (recency_weight * rec_score)
        scored.append((episode, combined))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:k]

# ── 工具方法 ────────────────────────────────────────────────

def _get_all_episodes(self) -> list[Episode]:
    return list(self.episodes)

def ___len__(self) -> int:
    return len(self.episodes)

def ___repr__(self) -> str:
    return f"EpisodicMemory({len(self.episodes)} episodes, strategy={self.boundary_strategy})"

EpisodicMemory.retrieve = _retrieve

EpisodicMemory.get_all_episodes = _get_all_episodes

EpisodicMemory.__len__ = ___len__

EpisodicMemory.__repr__ = ___repr__


### 第 4 步：把片段接入 LLM 提示词

当智能体收到新查询时，我们检索相关的过往片段，把它们的摘要注入系统提示词。这让 LLM 知道之前几个会话里发生过什么。


In [ ]:
def build_system_prompt(base_prompt: str, episodes: list[tuple]) -> str:
    """构建一个包含已检索片段摘要的系统提示词。"""
    if not episodes:
        return base_prompt

    episode_context = "\n\n".join(
        f"--- Episode from {ep.start_time[:16]} (topics: {', '.join(ep.topic_tags)}) ---\n"
        f"{ep.summary}"
        for ep, _score in episodes
    )

    return (
        f"{base_prompt}\n\n"
        f"## Relevant past episodes\n\n"
        f"{episode_context}"
    )


def chat_with_episodic_memory(
    memory: EpisodicMemory,
    user_input: str,
    session_messages: list[dict],
    base_system_prompt: str = "You are a helpful assistant with access to past conversation episodes.",
) -> str:
    """把带片段上下文的消息发给 LLM。"""
    # 检索相关的过往片段
    relevant_episodes = memory.retrieve(user_input, k=3)

    # 构建带片段摘要的系统提示词
    system_prompt = build_system_prompt(base_system_prompt, relevant_episodes)

    # 把新的用户消息加入会话
    session_messages.append({"role": "user", "content": user_input})

    response = llm.invoke([
            {"role": "system", "content": system_prompt},
            *session_messages,
        ])

    assistant_text = to_text(response.content).strip()
    session_messages.append({"role": "assistant", "content": assistant_text})

    # 把消息记入情景记忆缓冲区
    memory.add_message("user", user_input)
    memory.add_message("assistant", assistant_text)

    return assistant_text

## 运行示例

我们来模拟一个真实场景：一个项目教练智能体，在几天里跨三个会话与用户对话。每个会话结束后，我们把片段定稿。然后在最后一个会话里，测试智能体能否回忆起过往片段。

### 会话 1：项目启动


In [ ]:
memory = EpisodicMemory(boundary_strategy="session")

# --- 会话 1：项目启动 ---
session1_messages = []
base_prompt = (
    "You are a project-coaching assistant. Keep replies concise (2-3 sentences). "
    "Reference past episodes when relevant."
)

exchanges_s1 = [
    "Hi! I'm starting a new project. We're building a recommendation engine for an e-commerce platform.",
    "The team has 4 engineers. We're using Python and PostgreSQL. Timeline is 3 months.",
    "For the first milestone, we want a basic collaborative filtering model ready in 3 weeks.",
]

print("=" * 60)
print("SESSION 1: Project Kickoff")
print("=" * 60)

for msg in exchanges_s1:
    print(f"\n  User:  {msg}")
    reply = chat_with_episodic_memory(memory, msg, session1_messages, base_prompt)
    print(f"  Agent: {reply}")

# 定稿会话 1
ep1 = memory.end_session()
print(f"\n  [Episode finalized] {ep1}")
print(f"  Summary: {ep1.summary}")
print(f"  Tags: {ep1.topic_tags}")

### 会话 2：进展检查

几天后，用户回来讨论进展。注意 `session2_messages` 从空列表开始（全新会话）。智能体靠存储的片段恢复上下文。


In [ ]:
# --- 会话 2：进展检查 ---
session2_messages = []  # 全新会话，没有消息历史

exchanges_s2 = [
    "Hey, quick update. The collaborative filtering prototype is working. We hit 78% precision on the test set.",
    "But we're running into cold-start issues with new users. Any suggestions?",
    "Good idea. We'll try content-based fallback for new users with fewer than 5 interactions.",
]

print("=" * 60)
print("SESSION 2: Progress Check")
print("=" * 60)

for msg in exchanges_s2:
    print(f"\n  User:  {msg}")
    reply = chat_with_episodic_memory(memory, msg, session2_messages, base_prompt)
    print(f"  Agent: {reply}")

# 定稿会话 2
ep2 = memory.end_session()
print(f"\n  [Episode finalized] {ep2}")
print(f"  Summary: {ep2.summary}")
print(f"  Tags: {ep2.topic_tags}")

### 会话 3：回忆测试

现在用户让智能体回忆过往会话里发生的事。这个会话没有任何对话历史，智能体完全依赖情景记忆。


In [ ]:
# --- 会话 3：回忆测试 ---
session3_messages = []  # 又是全新会话

recall_questions = [
    "What project am I working on and what's the team size?",
    "What was the precision score we achieved on the collaborative filtering model?",
    "What problem did we identify with new users, and what was the proposed solution?",
]

print("=" * 60)
print("SESSION 3: Recall Test")
print("=" * 60)

for msg in recall_questions:
    print(f"\n  User:  {msg}")
    reply = chat_with_episodic_memory(memory, msg, session3_messages, base_prompt)
    print(f"  Agent: {reply}")

print("\n" + "=" * 60)
print(f"Total episodes stored: {len(memory)}")
print("=" * 60)

### 检查片段库

我们看看里面存了什么。每个片段有时间范围、标签和摘要。检索引擎搜索的就是这些内容。


In [ ]:
print("All stored episodes:\n")
for i, ep in enumerate(memory.get_all_episodes()):
    print(f"Episode {i + 1}:")
    print(f"  ID:       {ep.episode_id[:12]}...")
    print(f"  Time:     {ep.start_time[:16]} to {ep.end_time[:16]}")
    print(f"  Messages: {len(ep.messages)}")
    print(f"  Tags:     {ep.topic_tags}")
    print(f"  Summary:  {ep.summary}")
    print()

### 直接测试检索

我们看看检索引擎对不同查询怎么给片段打分。关于「冷启动」的查询应该让会话 2 排在前面；关于「项目搭建」的查询应该让会话 1 排在前面。


In [ ]:
test_queries = [
    "What was the cold-start problem we discussed?",
    "Tell me about the project setup and team.",
    "What technology stack are we using?",
]

for query in test_queries:
    print(f"Query: \"{query}\"")
    results = memory.retrieve(query, k=3)
    for ep, score in results:
        print(f"  Score {score:.3f} | tags={ep.topic_tags} | {ep.summary[:80]}...")
    print()

## 持久化

生产环境的情景记忆必须能在进程重启后存活。每个 `Episode` 都是一个 dataclass，所以我们可以把整个库序列化成 JSON。


In [ ]:
def save_episodes(memory: EpisodicMemory, path: str) -> None:
    """把所有片段保存到 JSON 文件。"""
    data = [ep.to_dict() for ep in memory.episodes]
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Saved {len(data)} episodes to {path}")


def load_episodes(path: str, boundary_strategy: str = "session") -> EpisodicMemory:
    """从 JSON 文件加载片段到一个新的 EpisodicMemory。"""
    with open(path) as f:
        data = json.load(f)
    mem = EpisodicMemory(boundary_strategy=boundary_strategy)
    mem.episodes = [Episode.from_dict(d) for d in data]
    print(f"Loaded {len(mem.episodes)} episodes from {path}")
    return mem


# 演示完整的存取往返
save_episodes(memory, "episodes.json")
loaded_memory = load_episodes("episodes.json")

# 验证加载后的记忆可用
results = loaded_memory.retrieve("What is the project about?", k=2)
for ep, score in results:
    print(f"  Retrieved: score={score:.3f} | {ep.summary[:80]}...")

## 权衡

### 情景记忆何时表现良好

- **多会话智能体**：用户期望智能体记得上次发生了什么。教练型智能体、项目跟踪工具、定期会议助手都能受益。
- **时序推理**：当用户问「上周我们定了什么？」「这件事随时间怎么变化的？」，片段时间戳让这类查询成为可能。
- **问责与审计**：每个片段都是一份完整记录。你可以回查智能体在什么时间点知道什么。

### 什么时候会出问题

- **边界检测很难**。基于会话的边界简单但粗糙（一个长会话变成一个巨大片段）。基于主题的检测需要额外的 LLM 调用，还可能误判。
- **存储随每个会话增长**。长期运行的智能体会积累成百上千个片段。没有修剪或归档，检索会变慢，成本会上升。
- **摘要丢失细节**。摘要抓住大意，但细粒度的事实可能丢失。如果用户问起三个会话前的某个具体数字，摘要里可能没有。
- **检索增加延迟**。每条用户消息都会触发一次嵌入调用，外加一次全量片段扫描。片段达到数千个时，你需要向量数据库，而不是线性扫描。

### 与其他记忆模式的对比

| 维度 | 缓冲记忆 | 摘要记忆 | 情景记忆 |
|--------|--------------|----------------|----------------|
| **范围** | 单个会话 | 单个会话 | 跨会话 |
| **粒度** | 每条消息 | 压缩摘要 | 完整片段 |
| **时间感知** | 只有轮次顺序 | 无 | 完整时间戳 |
| **检索** | 全部发送 | 发送摘要 | 按相关性搜索 |
| **最适合** | 短对话 | 长的单个会话 | 多会话智能体 |


## 延伸阅读

- [Tulving, "Episodic Memory: From Mind to Brain," *Annual Review of Psychology*, 2002](https://doi.org/10.1146/annurev.psych.53.100901.135114)：定义情景记忆、并把情景记忆与语义记忆区分开来的奠基性论文。
- [Park et al., "Generative Agents: Interactive Simulacra of Human Behavior," UIST 2023](https://arxiv.org/abs/2304.03442)：展示了拥有情景记忆的智能体，它们在一个模拟小镇中记住并反思过往经历。
- [Letta (MemGPT)](https://github.com/cpacker/MemGPT)：分层记忆管理的开源实现，包含面向 LLM 智能体的情景回忆。
- [Conway, "Episodic Memories," *Neuropsychologia*, 2009](https://doi.org/10.1016/j.neuropsychologia.2009.02.003)：从认知科学视角看情景记忆在人脑中如何组织和检索。
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings?utm_source=nirdiamant&utm_medium=github&utm_campaign=agent_memory_techniques)：本 notebook 语义检索所用嵌入 API 的参考文档。


---

*\u2190 上一篇：[08 - 知识图谱记忆](../08_knowledge_graph_memory/) \u00b7 下一篇：[10 - 语义记忆](../10_semantic_memory/) \u2192*

清理演示过程中创建的临时文件。


In [ ]:
for f in ["episodes.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Removed {f}")

## 🧪 动手试一试

三个小挑战，加深理解。每个大约需要 10-30 分钟。

### 挑战 1：基于嵌入的边界检测
把 `_detect_topic_shift()` 换成一个新版本：计算最新一条消息的嵌入与当前片段缓冲区平均嵌入之间的余弦距离，距离超过阈值就开启新片段。对比这种做法和基于 LLM 的方案的检测准确率。

### 挑战 2：检索权重调优
`retrieve()` 方法把语义相似度和新近度混合在一起。把新近度权重从 0.0 到 1.0、以 0.2 为步长逐档调整。对每个取值跑 5 个检索查询，记录返回了哪些片段。画出不同取值之间结果的重叠度，看看检索对权重有多敏感。

### 挑战 3：跨会话片段关联
用 `end_session()` 结束一个会话后，再开始一个新会话。用 `save_episodes()` 和 `load_episodes()` 把片段持久化。在新会话里，检索上一个会话的相关片段，把它们当作上下文使用。这是把情景记忆和 21 跨会话记忆的持久化策略直接结合起来。


![](https://europe-west1-amt-views-tracker.cloudfunctions.net/amt-tracker?notebook=all-techniques--09-episodic-memory--episodic-memory)
